# 8. Hybrid Search with Reciprocal Rank Fusion (RRF)

**RAG Pipeline Series — Notebook 8**

Notebook 7 put BM25 (sparse/keyword) and Chroma (dense/semantic) retrieval behind the same `.invoke()` interface and showed they fail on *different* queries: BM25 wins on exact-wording queries, dense retrieval wins on paraphrases. **Hybrid search** doesn't pick one — it runs both and merges their rankings, so a query only needs to succeed with *one* of the two strategies to rank well overall.

The standard way to merge two differently-scored ranked lists is **Reciprocal Rank Fusion (RRF)**. RRF ignores the raw scores entirely (BM25 scores and cosine-similarity scores aren't even on the same scale) and works purely from each list's **rank order**:

```
RRF_score(doc) = sum over each retriever r that returned doc of  1 / (k + rank_r(doc))
```

- `rank_r(doc)` is `doc`'s 1-indexed position in retriever `r`'s ranked results (a document missing from a list simply contributes 0 from that retriever).
- `k` is a small constant (60 is the standard default, from the original RRF paper) that flattens the curve — without it, the #1 result would dominate the score far more than is usually desirable.
- Documents that rank well in *multiple* retrievers accumulate score from each one, so genuine agreement between a lexical and a semantic signal is rewarded.

In this notebook we will:
1. Rebuild the BM25 and dense retrievers from notebook 7.
2. Implement RRF from scratch and fuse their two ranked lists by hand.
3. Reproduce the same fusion with LangChain's built-in `EnsembleRetriever`.
4. Compare BM25-only, dense-only, and hybrid rankings on the keyword-style and paraphrased queries from notebooks 3-7 — plus a query designed so *neither* retriever alone ranks the target chunk highly.

## Setup

In [ ]:
%pip install -q -U langchain langchain-classic langchain-community langchain-core rank_bm25 sentence-transformers langchain-huggingface langchain-chroma chromadb pandas

## 1. Recap: the two retrievers from notebook 7

Same chapter-tagged chunks, same `BM25Retriever`, same Chroma-backed dense retriever.

In [ ]:
from rag_utils import maybe_colab_upload

# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
maybe_colab_upload()

In [1]:
from rag_utils import build_chroma_store, get_embedder, load_chapter_chunks
from langchain_community.retrievers import BM25Retriever

pages, full_text, chapters, chunks = load_chapter_chunks()

keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 10

embeddings = get_embedder()
vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

print(f"{len(chunks)} chunks indexed into both retrievers")

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:03<00:00, 13.90it/s]


181 chunks indexed into both retrievers


## 2. Reciprocal Rank Fusion, from scratch

Given a list of ranked-document-lists (one per retriever), score every document that appears in *any* of them by summing `1 / (k + rank)` across the lists it appears in, then sort by total score. A document that's `#1` in one list and absent from the other still does well; a document that's mediocre (say rank 5-8) in *both* lists can out-score a document that's `#1` in only one, if the constant `k` is small enough relative to the rank gap — that's the "agreement" effect hybrid search is after.

In [2]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """ranked_lists: list of List[Document], each already sorted best-first by its own retriever."""
    scores, doc_lookup = {}, {}
    for ranked_docs in ranked_lists:
        for rank, doc in enumerate(ranked_docs, start=1):
            key = doc.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
            doc_lookup[key] = doc
    fused = sorted(scores.items(), key=lambda kv: -kv[1])
    return [(doc_lookup[key], round(score, 5)) for key, score in fused]


def show_fused(query, k=60, top_n=5):
    keyword_docs = keyword_retriever.invoke(query)
    dense_docs = dense_retriever.invoke(query)
    fused = reciprocal_rank_fusion([keyword_docs, dense_docs], k=k)
    print(f"query: {query!r}")
    for doc, score in fused[:top_n]:
        print(f"  rrf_score={score}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")
    print()

## 3. Fusing the notebooks 3-7 example queries

Same keyword-style and paraphrased queries used throughout this series. Neither retriever alone is "wrong" here — RRF just blends whichever one is already doing well with a second opinion from the other.

In [3]:
keyword_query = "Okapi Best Match 25 term frequency saturation document length normalization"
paraphrase_query = "How can giving a language model outside documents stop it from making things up?"

show_fused(keyword_query)
show_fused(paraphrase_query)

query: 'Okapi Best Match 25 term frequency saturation document length normalization'
  rrf_score=0.03279  chapter=02 (Evolution of Retrieval)
  rrf_score=0.02988  chapter=04 (Embeddings)
  rrf_score=0.02921  chapter=10 (Generation)
  rrf_score=0.01613  chapter=02 (Evolution of Retrieval)
  rrf_score=0.01613  chapter=05 (Vector Databases & Indexing)

query: 'How can giving a language model outside documents stop it from making things up?'
  rrf_score=0.03151  chapter=01 (Introduction to RAG)
  rrf_score=0.02964  chapter=09 (Augmentation)
  rrf_score=0.01639  chapter=10 (Generation)
  rrf_score=0.01639  chapter=03 (Data Ingestion)
  rrf_score=0.01613  chapter=01 (Introduction to RAG)



## 4. A query where *neither* retriever alone is strong

Consider a query that mixes a paraphrased concept with an exact technical term from a different part of the book — the kind of query that trips up a single-strategy retriever but is exactly what hybrid search is for.


In [4]:
mixed_query = "picking chunk boundaries so a passage stays semantically whole, not cut by character count"

keyword_docs = keyword_retriever.invoke(mixed_query)
dense_docs = dense_retriever.invoke(mixed_query)

print("BM25 alone:")
for doc in keyword_docs[:5]:
    print(f"  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

print("\nDense alone:")
for doc in dense_docs[:5]:
    print(f"  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

print()
show_fused(mixed_query)

BM25 alone:
  chapter=03 (Data Ingestion)
  chapter=03 (Data Ingestion)
  chapter=03 (Data Ingestion)
  chapter=10 (Generation)
  chapter=01 (Introduction to RAG)

Dense alone:
  chapter=03 (Data Ingestion)
  chapter=03 (Data Ingestion)
  chapter=03 (Data Ingestion)
  chapter=03 (Data Ingestion)
  chapter=03 (Data Ingestion)

query: 'picking chunk boundaries so a passage stays semantically whole, not cut by character count'
  rrf_score=0.03252  chapter=03 (Data Ingestion)
  rrf_score=0.032  chapter=03 (Data Ingestion)
  rrf_score=0.03102  chapter=03 (Data Ingestion)
  rrf_score=0.02991  chapter=03 (Data Ingestion)
  rrf_score=0.02899  chapter=01 (Introduction to RAG)



## 5. The same fusion via LangChain's `EnsembleRetriever`

`langchain.retrievers.EnsembleRetriever` implements weighted RRF internally — pass it a list of retrievers and a matching list of `weights` (must sum to 1) and it fuses their results the same way `reciprocal_rank_fusion()` did above, but as a drop-in `Retriever` you can hand to the rest of a chain like any other.

In [5]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[keyword_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

for doc in hybrid_retriever.invoke(paraphrase_query)[:5]:
    print(f"chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

chapter=01 (Introduction to RAG)
chapter=09 (Augmentation)
chapter=10 (Generation)
chapter=03 (Data Ingestion)
chapter=01 (Introduction to RAG)


Weights let you lean the fusion toward one retriever without dropping the other entirely — useful once you've measured (notebook 9) that one strategy is more reliable than the other for your specific document and query mix.

In [6]:
keyword_heavy = EnsembleRetriever(retrievers=[keyword_retriever, dense_retriever], weights=[0.8, 0.2])
dense_heavy = EnsembleRetriever(retrievers=[keyword_retriever, dense_retriever], weights=[0.2, 0.8])

print("80% BM25 / 20% dense, on the paraphrase query:")
for doc in keyword_heavy.invoke(paraphrase_query)[:5]:
    print(f"  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

print("\n20% BM25 / 80% dense, on the same query:")
for doc in dense_heavy.invoke(paraphrase_query)[:5]:
    print(f"  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")

80% BM25 / 20% dense, on the paraphrase query:
  chapter=01 (Introduction to RAG)
  chapter=09 (Augmentation)
  chapter=10 (Generation)
  chapter=01 (Introduction to RAG)
  chapter=01 (Introduction to RAG)

20% BM25 / 80% dense, on the same query:
  chapter=01 (Introduction to RAG)
  chapter=09 (Augmentation)
  chapter=03 (Data Ingestion)
  chapter=09 (Augmentation)
  chapter=03 (Data Ingestion)


## 6. Rank comparison: BM25 vs. dense vs. hybrid

Same rank-of-correct-chunk check used throughout this series, now with the hybrid retriever as a third column.

In [7]:
import pandas as pd

bm25_chunk_idx = next(i for i, d in enumerate(chunks) if "Okapi Best Match 25" in d.page_content)
hallucination_chunk_idx = next(i for i, d in enumerate(chunks) if "dynamic, external knowledge source" in d.page_content)


def rank_of(target_idx, retriever, query):
    docs = retriever.invoke(query)
    for rank, doc in enumerate(docs, start=1):
        if doc.page_content == chunks[target_idx].page_content:
            return rank
    return f"> {len(docs)}"


rows = [
    {
        "query": "Keyword-style",
        "bm25_rank": rank_of(bm25_chunk_idx, keyword_retriever, keyword_query),
        "dense_rank": rank_of(bm25_chunk_idx, dense_retriever, keyword_query),
        "hybrid_rank": rank_of(bm25_chunk_idx, hybrid_retriever, keyword_query),
    },
    {
        "query": "Paraphrased",
        "bm25_rank": rank_of(hallucination_chunk_idx, keyword_retriever, paraphrase_query),
        "dense_rank": rank_of(hallucination_chunk_idx, dense_retriever, paraphrase_query),
        "hybrid_rank": rank_of(hallucination_chunk_idx, hybrid_retriever, paraphrase_query),
    },
]
pd.DataFrame(rows)

,query,bm25_rank,dense_rank,hybrid_rank
0,Keyword-style,1,1,1
1,Paraphrased,2,> 10,5


## Takeaways

- RRF fuses ranked lists using rank position only, not raw scores — which sidesteps the problem of BM25 scores and cosine similarities living on completely different scales.
- The damping constant `k` (60 by default) controls how much a #1 rank dominates; smaller `k` sharpens the effect of top ranks, larger `k` flattens it.
- A document that both retrievers agree on (even at middling individual ranks) can outrank a document only one retriever loves — this is what lets hybrid search recover from either strategy's individual blind spots.
- LangChain's `EnsembleRetriever` is the same idea as a drop-in retriever, with `weights` to bias the fusion toward whichever strategy you trust more for a given corpus.
- Hybrid search is a strict improvement in *coverage* (fewer queries where both strategies fail simultaneously), not a guarantee that every individual query improves — measuring that properly needs the retrieval metrics built in the next notebook.

**Next up (notebook 9):** measuring retrieval quality with **Precision@k, Recall@k, MRR, and nDCG**, so "hybrid seems better" can be backed by numbers instead of a handful of example queries.